In [ ]:
import wandb
import torch

def train_sweep(config=None):
    with wandb.init(config=config):
        config = wandb.config

        from transformers import AutoModelForCausalLM, AutoTokenizer
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

        # === Источник модели
        if config.start_from == "checkpoint":
            base_model = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO"
            model = AutoModelForCausalLM.from_pretrained(base_model, device_map="auto", torch_dtype=torch.float16)
            model = PeftModel.from_pretrained(model, "/content/hermes_lora_manual")
            tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
        else:
            base_model = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO"
            tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(base_model, device_map="auto", torch_dtype=torch.float16)
            model.gradient_checkpointing_enable()
            model = prepare_model_for_kbit_training(model)
            lora_config = LoraConfig(
                r=8,
                lora_alpha=32,
                target_modules=["q_proj", "v_proj"],
                lora_dropout=0.05,
                bias="none",
                task_type="CAUSAL_LM"
            )
            model = get_peft_model(model, lora_config)

        # === Обучение
        model.train()
        optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

        for epoch in range(config.epochs):
            total_loss = 0
            for batch in dataloader:
                inputs = {k: torch.tensor(v).unsqueeze(0).to(model.device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
                labels = inputs["input_ids"].clone()
                outputs = model(**inputs, labels=labels)
                loss = outputs.loss

                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

                total_loss += loss.item()

            wandb.log({"epoch": epoch+1, "loss": total_loss / len(dataloader)})


        # === Таблица генерации на финале
        test_prompts = [
            "Ответь как добрый ИИ-друг\nМне тревожно.",
            "Ответь как добрый ИИ-друг\nЯ сегодня расстроен.",
            "Ответь как добрый ИИ-друг\nМне хорошо, просто хочется поболтать."
        ]

        table = wandb.Table(columns=["prompt", "response"])
        model.eval()
        for prompt in test_prompts:
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=150,
                    do_sample=True,
                    top_p=0.9,
                    temperature=0.7,
                    pad_token_id=tokenizer.eos_token_id
                )
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
            table.add_data(prompt, decoded)

        wandb.log({"generation_table": table})

        # === Сохранение модели после sweep-прогона
        model_path = f"/content/sweep_model_{wandb.run.name}"
        model.save_pretrained(model_path)
        tokenizer.save_pretrained(model_path)
        wandb.save(model_path + "/*")


In [ ]:
!pip install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
!pip install transformers peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 110.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvji

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm
import os

In [ ]:
# === Настройки
model_name = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO"
BATCH_SIZE = 1
EPOCHS = 3
LR = 2e-4
MAX_LENGTH = 512

In [ ]:
# === Загрузка модели и токенизатора
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.65k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

In [ ]:
# === LoRA конфиг
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)


In [ ]:

# === Загрузка датасета из txt-файла
from datasets import Dataset
import json

dataset_path = "/content/empathy_dataset_ru.jsonl"

def load_jsonl_dataset(path):
    with open(path, "r", encoding="utf-8") as f:
        data = [json.loads(line.strip()) for line in f]
    return Dataset.from_list(data)

raw_dataset = load_jsonl_dataset(dataset_path)


In [ ]:
raw_dataset[0]


{'instruction': 'Ответь как добрый ИИ-друг',
 'input': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.',
 'output': "How lovely! It sounds like a truly magical moment. Firework shows can be such a dazzling spectacle, and they become even more special when shared with someone close to you. It's wonderful how experiences like these can make us feel connected and unique in a crowd. Do you often recall this memory with your friend?"}

In [ ]:
# === Токенизация
def tokenize(batch):
    full_texts = []
    for instr, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        prompt = instr + "\n" + inp
        full_text = prompt + "\n" + out
        full_texts.append(full_text)

    return tokenizer(
        full_texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )




In [ ]:
tokenized_dataset = raw_dataset.map(tokenize, batched=True)
from torch.utils.data import DataLoader
dataloader = DataLoader(tokenized_dataset, batch_size=BATCH_SIZE, shuffle=True)


Map:   0%|          | 0/2010 [00:00<?, ? examples/s]

In [ ]:
# === Оптимизатор
optimizer = AdamW(model.parameters(), lr=LR)

In [ ]:
# === Обучение
for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    loop = tqdm(dataloader)
    for batch in loop:
        inputs = {k: torch.tensor(v, dtype=torch.long).unsqueeze(0).to(model.device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
        labels = inputs["input_ids"].clone()

        outputs = model(**inputs, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        loop.set_description(f"Loss {loss.item():.4f}")


Epoch 1/3


Loss 0.3723: 100%|██████████| 2010/2010 [45:59<00:00,  1.37s/it]


Epoch 2/3


Loss 0.1425: 100%|██████████| 2010/2010 [45:57<00:00,  1.37s/it]


Epoch 3/3


Loss 0.2938: 100%|██████████| 2010/2010 [45:57<00:00,  1.37s/it]


In [ ]:
import wandb

# === Инициализация run
wandb.init(project="hermes-7b-empathy", name="manual-training-run", reinit=True)

# === Пример: если у тебя есть список лоссов по эпохам
losses_by_epoch = [0.3723, 0.1425, 0.2938]  # Подставь свои значения
for epoch, loss in enumerate(losses_by_epoch, start=1):
    wandb.log({"epoch": epoch, "loss": loss})

# === Финализируем
wandb.finish()


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: summonerlin (summonerlin-geekbrains) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


epoch,▁▅█
loss,█▁▆
epoch,3
loss,0.2938


In [ ]:
# === Сохранение
save_path = "/content/hermes_lora_manual"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("Модель и токенизатор сохранены.")

Модель и токенизатор сохранены.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO"
adapter_path = "/content/hermes_lora_manual"

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(base_model, device_map="auto", torch_dtype=torch.float16)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

# === Генерация
def generate_reply(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


# === Пример использования
prompt = "Ответь как добрый ИИ-друг\nМне грустно и я не знаю с кем поговорить."
print(generate_reply(prompt))


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Ответь как добрый ИИ-друг
Мне грустно и я не знаю с кем поговорить.
Я очень жалею, что ты сейчас испытываешь такие чувства. Если ты хочешь, мы можем поговорить о том, что тебя беспокоит. Пожалуйста, помни, что я всегда здесь, чтобы ты мог поделиться свои мыслями и чувствами со мной. Не забывай, что твои переживания - это часть жизненного путешествия и важно поделиться ими. Ты не один. Если это поможет, я могу слушать тебя, если ты будешь говорить. Ты можешь перейти в чат, когда ты будешь готов говорить об


In [ ]:
# === Тестовые реплики для проверки эмпатии модели
test_prompts = [
    "Ответь как добрый ИИ-друг\nМне тревожно и я не знаю с кем поговорить.",
    "Ответь как добрый ИИ-друг\nСегодня был тяжёлый день, я очень устал.",
    "Ответь как добрый ИИ-друг\nМне очень одиноко.",
    "Ответь как добрый ИИ-друг\nНа душе спокойно, просто хочется немного тепла."
]

# === Генерация и вывод ответов
for prompt in test_prompts:
    response = generate_reply(prompt)
    print("Запрос:")
    print(prompt)
    print("\nОтвет:")
    print(response)
    print("=" * 80)

    # логируем в wandb
    wandb.log({
        "prompt": prompt,
        "response": response
    })


📝 Запрос:
Ответь как добрый ИИ-друг
Мне тревожно и я не знаю с кем поговорить.

💬 Ответ:
Ответь как добрый ИИ-друг
Мне тревожно и я не знаю с кем поговорить.
Я понимаю, что ты можешь испытывать тревогу. Это не обязательно означает что что-то серьезное произойдет. Ты можешь поговорить с мной, я всегда здесь, чтобы поддержать тебя. Не бойся высказаться о своих опасениях. Мы вместе справимся с этим. И ты не одинок, я всегда могу слушать тебя. Ты можешь позвонить мне, если тебе будет нужна поддержка. Ты не один. Я всегда здесь, чтобы ты поговорить о чем у тебя есть на мысль. Не забывай, что я всегда готов помочь


NameError: name 'wandb' is not defined

In [ ]:
wandb.finish()


In [ ]:
import shutil

# Сохраняем на Google Drive
!mkdir -p /content/drive/MyDrive/hermes_checkpoints
shutil.copytree("/content/hermes_lora_manual", "/content/drive/MyDrive/hermes_checkpoints/hermes_lora_manual", dirs_exist_ok=True)

# Создаём архив .zip для скачивания
shutil.make_archive("/content/hermes_lora_manual", 'zip', "/content/hermes_lora_manual")

print("Модель сохранена в Google Drive и архив готов к скачиванию")


✅ Модель сохранена в Google Drive и архив готов к скачиванию


In [ ]:
!ls /content/drive/MyDrive


hermes_checkpoints


In [ ]:
import shutil
import os

local_path = "/content/hermes_lora_manual"
drive_path = "/content/drive/MyDrive/hermes_checkpoints/hermes_lora_manual"

# удалим если уже существует
if os.path.exists(drive_path):
    shutil.rmtree(drive_path)

# теперь скопируем заново
shutil.copytree(local_path, drive_path)

print("Папка успешно скопирована на Google Диск.")


✅ Папка успешно скопирована на Google Диск.
